# M14 — Beyond the Stopwatch
## Cross-sport movement intelligence and the Hidden Braking Advantage

**Research question.** Can reliable, interpretable movement features inspired by other sports (motorsport, alpine skiing, tennis / basketball, rugby / soccer, track sprinting) reveal hidden athletic capabilities that predict NFL movement performance (`nfl.q90_shared`) **beyond the official Combine measurements (M1)**, on the **same held-out athletes**?

**What is new in M14.** M13 showed that learned encoders and generic biomechanics did not reliably beat M1 and that position explains most of the target. M14 replaces generic features with **event-based, mechanistically interpretable** metrics — braking, cornering, cutting, speed retention, re-acceleration and the acceleration–speed profile — each declared in a registry with its formula, failure cases, reliability requirement and NFL hypothesis **before** any outcome is seen.

**How this notebook works**
- Every section runs inside `RUN.run_stage` (status, timing, traceback in `logs/`, ledger in `manifests/stages.json`). A failed section does not stop independent later sections.
- Every expensive transformation (event extraction, perturbation re-extraction, reliability bootstrap, the M14 ladder, the permutation null) is cached on Drive with a dependency signature and reloaded after a disconnect: rerun from the top.
- No neural network is trained. M12 inductive checkpoints are adopted read-only; M12 / M13 directories are never written.
- Every figure is saved as PNG / SVG / PDF with its plotted data in `figures/data/`.
- Status vocabulary: `validated` · `exploratory` · `inconclusive` · `blocked` · `invalid` (declared rules in `docs/nflbowl/M14_RESEARCH_SPEC.md`).

Output root: `/content/drive/MyDrive/NFLBowl/experimental_lab/m14_cross_sport_athletic_intelligence_v1`

In [ ]:
# Setup 1 — install nflbowl from GitHub (Torch is never installed: the Colab CUDA build is kept)
REPO = 'git+https://github.com/mybenkhadda/NFL_bowl.git@main'
%pip install -q {REPO}
%pip install -q --upgrade --force-reinstall --no-deps {REPO}
%pip install -q markdown

In [ ]:
# Setup 2 — version check (after a reinstall: Runtime → Restart session, then continue from here)
import nflbowl
print('nflbowl', nflbowl.__version__)
assert tuple(int(x) for x in nflbowl.__version__.split('.')[:2]) >= (0, 6), 'nflbowl < 0.6.0 is imported: restart the session and rerun from Setup 2'

In [ ]:
# Setup 3 — configuration (unknown keys are errors; M14 has no training switch)
CFG = {
    'experiment_name': 'm14_cross_sport_athletic_intelligence_v1',
    'bundle_dir': '/content/drive/MyDrive/nfl-big-data-bowl-2027/bundles/m12f_v1_clean',
    'drive_root': '/content/drive/MyDrive/NFLBowl',
    'm12_run_dir': '/content/drive/MyDrive/NFLBowl/experimental_lab/m12_comprehensive_lab_v1',
    'm12_experiment_name': 'm12_comprehensive_lab_v1',
    'm13_run_dir': '/content/drive/MyDrive/NFLBowl/experimental_lab/m13_outcome_transfer_v1',
    'm13_experiment_name': 'm13_outcome_transfer_v1',
    'stage_dir': '/content/nflbowl_stage',
    'precision': 'fp32',
    'seed': 42,
    'encoders': ['F2', 'F3', 'F4', 'F5', 'F6'],
    'combo_encoder': 'F4',            # pre-declared (M12 primary design), never chosen on outcomes
    'n_boot': 2000,
    'reliability_threshold': 0.6,
    'reliability_marginal': 0.4,
    'robustness_athletes': 150,
    'n_permutations': 20,
    'single_metric_models': True,
    'reuse_artifacts': True,
}
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('not running on Colab: every path in CFG must exist locally')
from nflbowl.lab.m14 import config as M14C
cfg = M14C.resolve(CFG)
print('RUN_DIR        ', cfg.run_dir)
print('M12 experiment ', cfg.m12_experiment_dir)
print('M13 experiment ', cfg.m13_experiment_dir)
print('event params   ', cfg.event_params())

In [ ]:
# Setup 4 — imports, run directory, white report helpers
import json, math
from pathlib import Path
import numpy as np
import polars as pl
from IPython.display import display, HTML, Markdown
from nflbowl import _backend
from nflbowl.lab import handoff, environment, biomech, transfer_fairness as TF, figures as F
from nflbowl.lab.stats import pseudonym, spearman
from nflbowl.lab.store import StageSkipped
from nflbowl.lab.m14 import (registry as RG, events as EV, metrics as MT, reliability as MR, novelty as NV, transfer as TR,
                             scouting as SC, reporting as RP, figures as MF)
from nflbowl.data.validation import M1_COLS, TRANSFER_TARGET

CFG13 = cfg.m13()                         # the M13 hand-off helpers run unchanged on the M14 experiment
RUN = M14C.M14Run(cfg.run_dir, cfg, local_tmp='/content/m14_tmp')
R = RUN.results
SALT = cfg.experiment_name
P = EV.EventParams.from_config(cfg)
NBOOT_FAST = max(200, min(cfg.n_boot, 1000))    # secondary bootstraps (reliability CIs, positions)
SIGCOLS, TW_SUM, TW_LONG, PAIRS = {}, pl.DataFrame(), pl.DataFrame(), pl.DataFrame()

def banner(part, title, question):
    display(HTML(RP.banner_html(part, title, question)))

def say(text, kind='note'):
    display(HTML(RP.callout_html(text, kind)))

def show(df, digits=4, max_rows=30, status=None):
    s = RP.styled(df, digits, max_rows, status)
    display(s if s is not None else Markdown('_no rows_'))

def check(cond, msg):
    # scientific safeguard: an invalid comparison stops the section (logged as failed by run_stage)
    if not cond:
        raise TF.ComparisonError(msg)

def fig(f, name, data, question, caption):
    return RUN.figure(f, name, data=data, question=question, caption=caption)

def save(df, name, family, note=''):
    if df is not None and getattr(df, 'width', 0):
        RUN.table(df, name, family, note)
    return df

def labels_for(ids):
    return {int(i): (pseudonym(int(i), SALT) if cfg.anonymize_ids else str(int(i))) for i in ids}

def label_col(c):
    return c.replace('m14c_', '').replace('__', ' · ')

def family_long(cols):
    cols = [c for c in cols if c in ATH.columns]
    return ATH.select('nfl_id', 'position_group', *cols).unpivot(index=['nfl_id', 'position_group'], on=cols, variable_name='column', value_name='value').drop_nulls('value')

def position_summary(cols):
    d = family_long(cols)
    return d.group_by('column', 'position_group').agg(pl.len().alias('n'), pl.col('value').median().alias('median'), pl.col('value').quantile(0.25).alias('q25'),
                                                      pl.col('value').quantile(0.75).alias('q75')).filter(pl.col('n') >= 10).sort('column', 'position_group')

print('run dir', RUN.root)

---
# Part A — Environment and reproducibility

M14 opens its own experiment (`RUN_DIR/checkpoints/experiments/m14_cross_sport_athletic_intelligence_v1`) under the identical science configuration (hash checked) and adopts the M12 inductive checkpoints read-only (the M13 experiment is searched as a second source). A checkpoint is valid only if bundle fingerprint, science hash, regime, fold identity and architecture match and the file loads. Encoders missing a fold are excluded — there is no transductive fallback and no training.

In [ ]:
banner('A1', 'Environment, bundle and provenance', 'Is this session bound to the same bundle, science configuration and precision as M12 / M13?')

def _a1(ctx):
    global exp, ENV
    ENV = handoff.environment(CFG13)
    checks, bsum = environment.validate_bundle(CFG13.lab())
    save(checks, 'bundle_checks', 'handoff')
    show(checks, status='status')
    check(checks.filter(pl.col('status') == 'fail').height == 0, 'bundle checks failed: M14 refuses to run on other data')
    exp = handoff.open_experiment(CFG13, RUN)
    rev = ENV.get('revision') or {}
    RUN.set_provenance(bundle_fingerprint=exp.bundle.fingerprint, science_hash=exp.cfg.science_hash(), science_seed=exp.cfg.science.seed,
                       m14_seed=cfg.seed, code_revision=rev.get('commit'), nflbowl=nflbowl.__version__, precision=cfg.precision,
                       event_params_hash=M14C.params_hash(P.to_dict()))
    RUN.write_config(cfg, ENV)
    RUN.save_json(P.to_dict(), 'event_parameters')
    return bsum

BSUM = RUN.run_stage('A1', _a1, note='environment + bundle + experiment', raise_errors=True)
print('experiment', exp.id, '· science', exp.cfg.science_hash(), '· bundle', exp.bundle.fingerprint, '· sequences', exp.bundle.n_sequences)

In [ ]:
banner('A2', 'Read-only adoption of the M12 inductive encoders', 'Which (encoder, fold) checkpoints are valid for reuse in the embedding comparisons?')

def _a2(ctx):
    folds = sorted(set(int(v) for v in exp.folds_by_athlete.values()))
    src = handoff.discover(CFG13, exp)
    save(src, 'checkpoint_sources', 'handoff')
    adopted = handoff.adopt(exp, src, list(cfg.encoders)) if src.height else pl.DataFrame()
    emb = handoff.adopt_embeddings(CFG13, exp) if cfg.adopt_embeddings else pl.DataFrame()
    inv = handoff.inventory(CFG13, exp, ['F0', *cfg.encoders], folds, verify_load=cfg.verify_checkpoint_load)
    ready, missing = handoff.ready_models(inv, folds)
    for name, df in (('adoption', adopted), ('embedding_adoption', emb), ('checkpoint_inventory', inv)):
        save(df, name, 'handoff')
    return folds, ready, missing, inv

FOLDS, READY, MISSING, INV = RUN.run_stage('A2', _a2, note='discover · adopt · inventory (no training)', raise_errors=True)
INV_PLOT = INV.with_columns(pl.col('ready_for_transfer').cast(pl.Float64).alias('ready'), pl.col('model').map_elements(F.short, return_dtype=pl.String))
fig(F.heatmap(INV_PLOT, 'model', 'fold', 'ready', 'Encoder readiness per outer fold', fmt='{:.0f}', vmin=0.0, vmax=1.0,
              subtitle='1 = valid inductive checkpoint + embedding route; encoders missing a fold are excluded'),
    'a2_readiness_matrix', {'inventory': INV}, 'Which encoders can enter the embedding comparisons?', 'Readiness after read-only adoption.')
print('ready on every fold:', READY, '· missing:', MISSING or 'none', '· combo encoder:', TR.combo_name(cfg.combo_encoder))
if TR.combo_name(cfg.combo_encoder) not in READY:
    say(f'The pre-declared combo encoder {cfg.combo_encoder} is not ready on every fold: the model M1 + reliable metrics + embedding is BLOCKED (no substitute is chosen).', 'warning')

In [ ]:
banner('A3', 'Checkpoint references and prior-milestone provenance', 'Which M12 / M13 artifacts does M14 build on (read-only)?')

def _a3(ctx):
    m12, m13 = Path(cfg.m12_run_dir), Path(cfg.m13_run_dir)
    refs = {'m12_run_dir': str(m12), 'm12_found': m12.is_dir(), 'm12_experiment': str(cfg.m12_experiment_dir),
            'm13_run_dir': str(m13), 'm13_found': m13.is_dir(), 'm13_experiment': str(cfg.m13_experiment_dir),
            'checkpoint_sources': [str(p) for p in handoff.source_dirs(CFG13)], 'ready_encoders': READY, 'missing_folds': MISSING,
            'combo_encoder': TR.combo_name(cfg.combo_encoder),
            'adopted': INV.filter(pl.col('ready_for_transfer').fill_null(False)).select('model', 'fold', 'imported_from').to_dicts() if 'imported_from' in INV.columns else [],
            'm13_tables': sorted(p.relative_to(m13).as_posix() for p in m13.glob('tables/*/*.parquet')) if m13.is_dir() else [],
            'note': 'M12 / M13 numbers are context only; no M14 result is read from them'}
    RUN.save_json(refs, 'checkpoint_references')
    return refs

REFS = RUN.run_stage('A3', _a3, note='references only')
if REFS:
    print('M12 found:', REFS['m12_found'], '· M13 found:', REFS['m13_found'], '· M13 tables available:', len(REFS['m13_tables']))
RUN.write_status()

---
# Part B — Dataset and measurement feasibility

The M12 bundle stores, per 10 Hz sequence: speed, tangential acceleration (signed), normal / centripetal acceleration (signed), heading rate, frame duration and a heading-valid flag, plus a validity mask. There are **no x / y positions, no body orientation and no synchronised opponents or ball**. Every registry metric is checked against these inputs; metrics needing missing inputs are **blocked** and never approximated.

The event extraction below is the single expensive pass of M14 (≈ 77k sequences). It is cached in `features/events/events.parquet` and `features/sequence/sequence_records.parquet`, keyed by the bundle fingerprint and the hash of the event parameters.

In [ ]:
banner('B1', 'Measurement feasibility and the extraction pass', 'Which cross-sport metrics can the available channels support, and what does the 10 Hz signal quality allow?')

def _b1(ctx):
    global AV, EVS, SEQS, QUAL
    b = exp.bundle
    check(int(b.X.shape[2]) == len(RG.CHANNELS), f'expected {len(RG.CHANNELS)} channels, found {b.X.shape[2]}: channel layout changed')
    acols = list(b.athletes.columns) if b.athletes is not None else []
    AV = RG.availability(RG.CHANNELS, athlete_columns=acols, synchronized=False)
    save(AV, 'metric_availability', 'feasibility')
    EVS, SEQS, how = MT.cached_extract(RUN, exp, P, reuse=cfg.reuse_artifacts)
    ctx['cache'] = 'hit' if how == 'reused cache' else None
    print('extraction:', how, '·', EVS.height, 'events ·', SEQS.height, 'sequences')
    QUAL = (SEQS.group_by('domain', 'kind').agg(pl.len().alias('n_sequences'), pl.col('n_valid').mean().alias('mean_valid_frames'),
                                               pl.col('n_segments').mean().alias('mean_segments'), (pl.col('n_dt_gaps') > 0).mean().alias('share_dt_gap'),
                                               (pl.col('n_internal_invalid') > 0).mean().alias('share_mask_hole'),
                                               pl.col('heading_valid_share').mean().alias('heading_valid'), pl.col('n_events').mean().alias('events_per_seq'),
                                               pl.col('n_complete_events').mean().alias('complete_events_per_seq')).sort('domain', 'kind'))
    save(QUAL, 'signal_quality_by_kind', 'feasibility')
    return how

RUN.run_stage('B1', _b1, note='availability + one-pass event extraction (cached)', raise_errors=True)
show(AV, status='status', max_rows=40)
show(QUAL, digits=3)
AV_SUM = AV.group_by('family', 'status').len().sort('family', 'status')
save(AV_SUM, 'availability_summary', 'feasibility')
QP = QUAL.with_columns((pl.col('domain') + ' · ' + pl.col('kind')).alias('label')).sort('events_per_seq')
fig(F.hbar(QP, 'label', 'events_per_seq', 'Detected cut events per sequence', 'mean events per sequence', fmt='{:.2f}',
           subtitle='agility drills and NFL braking / redirection windows should carry events; the forty should not'),
    'b1_events_per_sequence', {'quality': QUAL}, 'Does the event detector fire where cuts exist?', 'Mean detected events per sequence by domain and drill / window kind.')
n_blk = AV.filter(pl.col('status') == 'blocked').height
say(f'{AV.height - n_blk} metrics are computable and {n_blk} are blocked (missing inputs listed in the table). Blocked metrics are reported as blocked in every later section; nothing is approximated.', 'finding')

**Reading the feasibility audit.** A sequence with mask holes or long frame durations is split into gap-free segments; events never cross a segment boundary, so a hole inside a cut removes that cut rather than bridging it with interpolated values. At 10 Hz a phase boundary is resolved to about 0.1 s; transition times are linearly interpolated between frames, which is still limited by the provider's smoothing. Events per sequence near zero for the forty (straight line) and clearly positive for shuttles / three-cone and NFL braking windows is the face-validity check of the detector.

---
# Part C — Cross-sport metric definitions

The registry is the contract of M14: every metric's origin, formula, channels, unit, level, sampling assumption, event definition, failure cases, positions, reliability requirement, NFL hypothesis and sign are fixed here, before any outcome is seen. Saved as `tables/registry/metric_definitions.(parquet|csv)` and `tables/registry/metric_definitions.json`.

In [ ]:
banner('C1', 'The cross-sport metric registry', 'What exactly is each metric, where does it come from, and when is it undefined?')

def _c1(ctx):
    d = RG.definitions_frame()
    save(d, 'metric_definitions', 'registry')
    RUN.save_json(RG.definitions_json(), 'metric_definitions', subdir='tables/registry')
    RUN.save_json(P.to_dict(), 'event_parameters', subdir='tables/registry')
    return d

DEFS = RUN.run_stage('C1', _c1, note='registry → tables/registry')
show(DEFS.select('name', 'family', 'origin_sport', 'level', 'unit', 'formula', 'sign', 'braking_core'), max_rows=40)
show(DEFS.group_by('family', 'level').len().sort('family', 'level'))
for fam in RG.FAMILIES:
    sub = DEFS.filter(pl.col('family') == fam)
    display(Markdown(f'**{fam}** — ' + '; '.join(f'`{r[0]}`: {r[1]}' for r in sub.select('name', 'hypothesis').iter_rows())))
say('Every metric is a kinematic proxy: no force, power, tendon elasticity, reaction time or injury risk is inferred. Signs (+1 / −1) state the hypothesised better direction and are used only by the braking composite.', 'caveat')

---
# Part D — Movement-event extraction

Each cut is decomposed into **approach → braking onset → turn phase → minimum speed → propulsion onset → exit → recovery** (definitions in `nflbowl.lab.m14.events`). Event metrics are aggregated to a sequence (median over its events), then to an athlete (median over the sequences of a scope: `forty`, `agility` = three-cone + shuttles, `drill` = position drills). NFL windows produce `m14n__*` columns, which are **outcomes only** and never enter a predictor.

In [ ]:
banner('D1', 'Event anatomy, completeness and athlete signatures', 'What does a detected cut look like, how often is each phase observed, and how many athletes get a signature?')

def _d1(ctx):
    global EVX, ST, ATH
    prov = MT.extraction_signature(exp.bundle.fingerprint, P)
    EVX = MT.nan_to_null(EVS).with_columns(MT.scope_expr().alias('scope'))
    ST = RUN.load_data('features/sequence/sequence_table', prov) if cfg.reuse_artifacts else None
    ATH = RUN.load_data('features/athlete/athlete_features', prov) if cfg.reuse_artifacts else None
    if ST is None or ATH is None:
        ST = MT.sequence_table(EVS, SEQS, exp.folds_by_athlete)
        ATH = MT.athlete_table(ST, EVS)
        RUN.data_file(ST, 'features/sequence/sequence_table', prov)
        RUN.data_file(ATH, 'features/athlete/athlete_features', prov)
    else:
        ctx['cache'] = 'hit'
    comp = (EVX.group_by('domain', 'scope').agg(pl.len().alias('n_events'), pl.col('nfl_id').n_unique().alias('n_athletes'),
                                               pl.col('complete').mean().alias('share_complete'), pl.col('exit_censored').mean().alias('share_exit_censored'),
                                               pl.col('propulsion_censored').mean().alias('share_propulsion_censored'),
                                               pl.col('recovery_censored').mean().alias('share_recovery_censored'), pl.col('weak_braking').mean().alias('share_weak_braking'),
                                               pl.col('v_entry').median().alias('median_v_entry'), pl.col('v_min').median().alias('median_v_min'),
                                               pl.col('braking_duration').median().alias('median_braking_s'), pl.col('turn_angle').abs().median().alias('median_abs_turn_rad'))
            .sort('domain', 'scope'))
    save(comp, 'event_completeness', 'events')
    agree = (EVX.filter((pl.col('turn_dir') != 0) & (pl.col('an_turn_sign') != 0)).group_by('domain')
             .agg(pl.len().alias('n_directional_events'), (pl.col('turn_dir') == pl.col('an_turn_sign')).mean().alias('yaw_vs_an_sign_agreement')))
    save(agree, 'turn_sign_agreement', 'events')
    return comp, agree

COMP, AGREE = RUN.run_stage('D1', _d1, note='sequence + athlete tables (cached)', raise_errors=True)
show(COMP, digits=3)
show(AGREE, digits=3)
PROF_C = MF.aligned_profile(EVX.filter(pl.col('domain') == 'combine'), 'scope', P.trace_half_window)
fig(MF.event_anatomy(PROF_C, 'Anatomy of a Combine cut (complete events, median ± IQR)', 'aligned on the speed minimum; agility drills vs position drills'),
    'd1_event_anatomy_combine', {'profile': PROF_C}, 'What does braking → turn → propulsion look like at 10 Hz?', 'Median and IQR of speed and tangential acceleration around the minimum.')
PROF_D = MF.aligned_profile(EVX.filter(pl.col('scope').is_in(['agility', 'nfl'])), 'scope', P.trace_half_window)
fig(MF.event_anatomy(PROF_D, 'Combine agility cuts vs NFL braking / redirection windows', 'the domain shift of the same event, seen directly'),
    'd1_event_anatomy_domains', {'profile': PROF_D}, 'Do Combine cuts and NFL cuts share their shape?', 'Aligned profiles by scope (agility vs NFL).')
CAND = MT.combine_predictors(ATH)
print(len(CAND), 'Combine predictor columns ·', ATH.height, 'athletes')
if AGREE is not None and AGREE.height:
    say('Turn direction comes from the heading rate; its agreement with the sign of the centripetal acceleration is a sign-convention check. Asymmetry metrics use magnitudes only, so they do not depend on the convention.', 'method')

---
# Part E — Motorsport and braking mechanics

- **Corner Exit Efficiency** — speed 0.5 s after the minimum relative to the entry speed (does the athlete leave the corner fast?).
- **Braking Point Efficiency** — the constant deceleration that would shed the same speed over the measured braking distance, (v_b² − v_min²) / 2d. Late, hard braking gives high values. A work-energy equivalent, not a force.
- **Braking Budget Utilization** — mean / peak deceleration in the braking phase: a square brake trace (high) vs a spike (low).
- **Stop–Go Elasticity Proxy** — propulsion in the 0.5 s after the minimum relative to braking in the 0.5 s before (a proxy; no tendon inference).

In [ ]:
banner('E1', 'Motorsport metrics and the braking phase', 'How do braking and corner-exit mechanics vary across positions at the Combine?')

MOTO = [m.name for m in RG.METRICS if m.family == 'motorsport']
def _e1(ctx):
    cols = [RG.combine_column(m, 'agility') for m in MOTO]
    ps = position_summary(cols)
    save(ps, 'motorsport_by_position', 'metrics')
    phase = (EVX.filter(pl.col('complete')).group_by('domain', 'scope').agg(*[pl.col(c).median().alias(c) for c in ('braking_duration', 'braking_distance', 'peak_decel', 'mean_decel', *MOTO)])
             .sort('domain', 'scope'))
    save(phase, 'braking_phase_medians', 'metrics')
    return ps, phase

PS_E, PHASE_E = RUN.run_stage('E1', _e1)
show(PHASE_E, digits=3)
c_bpe = RG.combine_column('braking_point_efficiency', 'agility')
if c_bpe in ATH.columns:
    fig(F.distribution_compare(ATH.drop_nulls(c_bpe), c_bpe, 'position_group', 'Braking Point Efficiency by position (agility drills)', 'equivalent deceleration (yd/s²)'),
        'e1_bpe_by_position', {'athletes': ATH.select('nfl_id', 'position_group', c_bpe)}, 'Do positions differ in how late / hard they brake?',
        'Athlete medians of the work-energy equivalent deceleration.')
sev = EVX.filter(pl.col('complete') & (pl.col('scope') == 'agility')).select('peak_decel', 'braking_point_efficiency').drop_nulls()
sev = sev.sample(min(4000, sev.height), seed=cfg.seed) if sev.height else sev
fig(F.scatter(sev, 'peak_decel', 'braking_point_efficiency', 'Braking Point Efficiency vs single-frame peak deceleration', 'peak deceleration (yd/s²)',
              'equivalent deceleration (yd/s²)', subtitle='agility events (sample); below the identity = braking spread over a longer distance', identity=True),
    'e1_bpe_vs_peak', {'events': sev}, 'Is BPE more than a peak-deceleration restatement?', 'Event-level comparison of BPE and peak deceleration.')
if sev.height >= 30:
    r = spearman(sev['peak_decel'].to_numpy(), sev['braking_point_efficiency'].to_numpy())
    say(f'Event-level Spearman between peak deceleration and BPE: {r:.2f}. A value well below 1 means BPE carries the braking-distance information that a single-frame peak does not.', 'finding')

---
# Part F — Alpine skiing and directional-change efficiency

- **Turn Speed Retention** — v_min / v_entry.
- **Curvature-Speed Envelope** — q90 of centripetal acceleration in speed bins (2–4, 4–6, 6–8, ≥ 8 yd/s), averaged: a lateral-capacity envelope.
- **Multi-Cut Consistency** — SD of corner-exit efficiency across the cuts of one drill (lower = more repeatable).
- **Directional Symmetry** — magnitude of the left / right difference of corner-exit efficiency. Descriptive only, never injury risk.

In [ ]:
banner('F1', 'Speed retention, the curvature-speed envelope and symmetry', 'How much speed do athletes carry through turns, and how does lateral capacity change with speed?')

def _f1(ctx):
    tsr = EVX.filter(pl.col('complete') & (pl.col('scope') == 'agility')).select('v_entry', 'turn_speed_retention', 'position_group').drop_nulls()
    tsr = tsr.sample(min(4000, tsr.height), seed=cfg.seed) if tsr.height else tsr
    env_cols = [c for c in ST.columns if c.startswith('envelope_q90_')]
    env = (ST.with_columns(MT.scope_expr()).filter(pl.col('_scope') == 'agility').join(ATH.select('nfl_id'), on='nfl_id')
           .group_by('position_group').agg(*[pl.col(c).median() for c in env_cols], pl.len().alias('n')).filter(pl.col('n') >= 30)
           .unpivot(index=['position_group', 'n'], on=env_cols, variable_name='bin', value_name='q90_abs_a_n')
           .with_columns(pl.col('bin').str.extract(r'envelope_q90_(\d+)_', 1).cast(pl.Float64).alias('speed_bin_lo')).drop_nulls('q90_abs_a_n').sort('position_group', 'speed_bin_lo'))
    save(env, 'curvature_speed_envelope_by_position', 'metrics')
    sym = position_summary([RG.combine_column('directional_asymmetry', 'agility'), RG.combine_column('turn_speed_retention', 'agility'),
                            RG.combine_column('multi_cut_consistency', 'agility')])
    save(sym, 'alpine_by_position', 'metrics')
    return tsr, env, sym

TSR_E, ENV_P, SYM = RUN.run_stage('F1', _f1)
if TSR_E is not None and TSR_E.height >= 30:
    r = spearman(TSR_E['v_entry'].to_numpy(), TSR_E['turn_speed_retention'].to_numpy())
    fig(F.scatter(TSR_E, 'v_entry', 'turn_speed_retention', 'Turn speed retention falls with entry speed', 'entry speed (yd/s)', 'v_min / v_entry',
                  subtitle=f'agility events (sample) · Spearman {r:.2f} — the reason for the speed-adjusted cutting score'),
        'f1_tsr_vs_entry', {'events': TSR_E}, 'Is retention confounded by approach speed?', 'Event-level retention vs entry speed.')
if ENV_P is not None and ENV_P.height:
    fig(F.lines(ENV_P, 'speed_bin_lo', 'q90_abs_a_n', 'position_group', 'Curvature-speed envelope by position (agility drills)', 'speed bin lower edge (yd/s)',
                'q90 |centripetal acceleration| (yd/s²)'),
        'f1_envelope_by_position', {'envelope': ENV_P}, 'How does lateral capacity change with speed?', 'Position medians of the per-bin q90 centripetal acceleration.')
show(SYM, digits=3)
say('Asymmetry here is a movement description from 10 Hz heading data; it is never read as injury risk, and its sign convention is irrelevant (magnitudes only).', 'caveat')

---
# Part G — Court-sport transitions (tennis / basketball)

- **Braking-to-Propulsion Transition** — interpolated time from braking release (a_tan rising through −1 yd/s²) to propulsion (rising through +1 yd/s²). A movement transition, **not a reaction time** (no stimulus timestamp exists).
- **First-Step Explosiveness** — mean tangential acceleration in the first 0.3 s of propulsion.
- **Cut Recovery Time** — time from the minimum until 90 % of the entry speed is regained (stands in for lateral recovery, which is blocked: no lateral displacement / orientation channel).
- **SPARC** — spectral arc length of the speed profile (descriptive smoothness, not a mechanism).

In [ ]:
banner('G1', 'Transitions, first steps, recovery and smoothness', 'How quickly do athletes switch from braking to propulsion and regain speed?')

COURT = [m.name for m in RG.METRICS if m.family == 'court' and m.level != 'blocked']
def _g1(ctx):
    tr = EVX.filter(pl.col('complete') & pl.col('scope').is_in(['agility', 'drill', 'nfl'])).select('scope', 'brake_propulsion_transition', 'cut_recovery_time').drop_nulls()
    ps = position_summary([RG.combine_column(m, s) for m in COURT for s in RG.BY_NAME[m].scopes])
    save(ps, 'court_by_position', 'metrics')
    sp = ST.with_columns(MT.scope_expr()).group_by('domain', 'kind').agg(pl.col('sparc_smoothness').median().alias('median_sparc'), pl.col('sparc_smoothness').count().alias('n'))
    save(sp, 'sparc_by_kind', 'metrics')
    return tr, ps, sp

TR_G, PS_G, SP_G = RUN.run_stage('G1', _g1)
if TR_G is not None and TR_G.height:
    fig(F.histogram_by(TR_G, 'brake_propulsion_transition', 'scope', 'Braking-to-propulsion transition time', 'seconds (interpolated)', bins=40),
        'g1_transition_time', {'events': TR_G.sample(min(20000, TR_G.height), seed=cfg.seed)}, 'How long is the switch from braking to propulsion?',
        'Distribution by scope; 10 Hz resolution, linear interpolation between frames.')
c_fs = RG.combine_column('first_step_explosiveness', 'agility')
if c_fs in ATH.columns:
    d = ATH.select('nfl_id', 'position_group', c_fs).join(exp.bundle.athletes.select('nfl_id', 'ten'), on='nfl_id', how='left').drop_nulls()
    if d.height >= 30:
        r = spearman(d[c_fs].to_numpy(), d['ten'].to_numpy())
        fig(F.scatter(d, 'ten', c_fs, 'First-step explosiveness after a cut vs the official ten-yard split', 'ten-yard split (s)', 'mean a_tan, first 0.3 s (yd/s²)',
                      group='position_group', subtitle=f'Spearman {r:.2f}: a weak relation means the cut first step is not the start of a sprint'),
            'g1_first_step_vs_ten', {'athletes': d}, 'Is post-cut first-step acceleration a restatement of the ten split?', 'Athlete medians vs official ten split.')
show(SP_G, digits=3)

---
# Part H — Rugby, soccer and sprinting metrics

- **Change-of-Direction Deficit Residual** — official agility time minus what forty (+ weight) predicts, with the regression refitted on **training athletes of each outer fold** inside the transfer models. Shown here descriptively as out-of-fold residuals.
- **Speed-Adjusted Cutting Score** — retention beyond what entry speed predicts (train-fold OLS in the transfer models).
- **Maximum Controlled Deceleration** — highest 2-frame deceleration within completed cuts (suppresses single-frame spikes).
- **Acceleration–speed profile** (A0, V0, reserve at 7 yd/s), **Time to Useful Speed** (6 yd/s) and **High-Speed Acceleration** from the forty. Mass-free kinematic profiles: no horizontal force or power is claimed.

In [ ]:
banner('H1', 'COD deficit, controlled deceleration and the sprint profile', 'Do sprint-profile and change-of-direction metrics separate athletes with the same stopwatch times?')

def _h1(ctx):
    a = exp.bundle.athletes
    offs = [c for c in ('forty', 'ten', 'weight', 'three_cone', 'short_shuttle') if c in a.columns]
    d = ATH.join(a.select('nfl_id', *offs), on='nfl_id', how='left')
    d = d.with_columns(pl.Series('fold', [exp.folds_by_athlete.get(int(i), -1) for i in d['nfl_id'].to_list()]))
    rows = []
    for name, y, covs in NV.residual_specs(d.columns):
        r = NV.oof_residual(d, y, covs)
        d = d.with_columns(pl.Series(f'oof_{name}', r).fill_nan(None))
        rows.append({'metric': name, 'residualised': y, 'covariates': ', '.join(covs), 'n': int(np.isfinite(r).sum())})
    save(pl.DataFrame(rows) if rows else pl.DataFrame({'metric': ['cod_deficit_residual'], 'n': [0]}), 'residual_metrics_descriptive', 'metrics')
    sprint = [RG.combine_column(m.name, 'forty') for m in RG.METRICS if m.family == 'sprint']
    corr = []
    for c in sprint:
        for o in ('forty', 'ten'):
            if c in d.columns and o in d.columns:
                x = d.select(c, o).drop_nulls()
                corr.append({'metric': c, 'official': o, 'n': x.height, 'spearman': spearman(x[c].to_numpy(), x[o].to_numpy()) if x.height >= 30 else float('nan')})
    corr = pl.DataFrame(corr, infer_schema_length=None)
    save(corr, 'sprint_vs_officials', 'metrics')
    return d, corr

D_H, CORR_H = RUN.run_stage('H1', _h1)
show(CORR_H, digits=3)
ca0, cv0 = RG.combine_column('accel_envelope_a0', 'forty'), RG.combine_column('accel_envelope_v0', 'forty')
if D_H is not None and ca0 in D_H.columns and cv0 in D_H.columns:
    dd = D_H.select('nfl_id', 'position_group', ca0, cv0).drop_nulls()
    fig(F.scatter(dd, cv0, ca0, 'Acceleration–speed profile of the forty: A0 vs V0', 'V0 (yd/s, extrapolated)', 'A0 (yd/s²)', group='position_group',
                  subtitle='athletes with the same forty can reach it with different A0 / V0 balances'),
        'h1_av_profile', {'athletes': dd}, 'Do athletes with equal forty times differ in their acceleration profile?', 'Athlete medians of the linear a–v fit.')
oc = [c for c in (D_H.columns if D_H is not None else []) if c.startswith('oof_cod_deficit')]
if oc:
    show(D_H.select('position_group', *oc).group_by('position_group').agg(*[pl.col(c).median() for c in oc], pl.len().alias('n')).sort('position_group'), digits=3)
else:
    say('No official agility time column (three_cone / short_shuttle) in the athletes table: the COD-deficit residual is BLOCKED.', 'warning')

---
# Part I — Reliability and feature quality

A metric is promoted only if it is **measured reliably**. Trials are Combine sequences (attempts); agility columns pool three drills and are kind-centred. Reported: ICC(1,1) with an athlete-cluster bootstrap CI, ICC(1,k̄) of the athlete median, split-half with Spearman–Brown, ICC at **equal trial counts** on a fixed athlete set, the projected reliability versus number of trials, robustness to noise and smoothing (re-extraction of a seeded athlete sample), and coverage. The measurement status is declared in `nflbowl.lab.m14.reliability`; unstable metrics are flagged `invalid`, not promoted.

In [ ]:
banner('I1', 'Trial reliability, equal-trial-count curves and split-half', 'Which metrics rank athletes consistently from one attempt to the next?')

def _i1(ctx):
    global CACHE
    CACHE = MR.trial_cache(ST, CAND)
    sig = {'extraction': MT.extraction_signature(exp.bundle.fingerprint, P), 'cand': sorted(CAND), 'n_boot': NBOOT_FAST, 'seed': cfg.seed,
           'k': list(cfg.reliability_k_grid), 'rep': cfg.reliability_repeats, 'min': cfg.min_reliability_athletes}
    hit = RUN.cached('reliability', ['reliability_trials', 'reliability_matched_k', 'reliability_projected', 'reliability_athlete_level'], sig, 'reliability/core')
    if hit is not None:
        ctx['cache'] = 'hit'
        return hit['reliability_trials'], hit['reliability_matched_k'], hit['reliability_projected'], hit['reliability_athlete_level']
    rel = MR.reliability_table(CACHE, NBOOT_FAST, cfg.seed, cfg.min_reliability_athletes)
    mk = MR.matched_k_curves(CACHE, cfg.reliability_k_grid, cfg.reliability_repeats, cfg.seed, cfg.min_reliability_athletes)
    proj = MR.projected_curve(rel)
    asym = [c for c in CAND if RG.parse_column(c)[1] in RG.ATHLETE_METRICS]
    rela = MR.athlete_level_split_half(ST, EVS, asym)
    RUN.save_all({'reliability_trials': rel, 'reliability_matched_k': mk, 'reliability_projected': proj, 'reliability_athlete_level': rela}, 'reliability', sig, 'reliability/core')
    return rel, mk, proj, rela

REL_T, MK, PROJ, REL_A = RUN.run_stage('I1', _i1, note='ICC · bootstrap · matched-k · split-half (cached)', raise_errors=True)
CACHE = CACHE if 'CACHE' in globals() and CACHE else MR.trial_cache(ST, CAND)
show(REL_T.select('column', 'n_athletes', 'n_obs', 'k_bar', 'icc1', 'icc1_lo', 'icc1_hi', 'icc_k', 'split_r', 'split_sb').sort('icc1', descending=True), digits=3, max_rows=60)
RP_ = REL_T.filter(pl.col('icc1').is_not_null() & pl.col('icc1').is_not_nan()).with_columns(pl.col('column').map_elements(label_col, return_dtype=pl.String).alias('label')).sort('icc1')
fig(F.forest(RP_, 'label', 'icc1', 'icc1_lo', 'icc1_hi', 'Trial reliability of every M14 metric', 'ICC(1,1) with athlete-bootstrap 95 % CI',
             ref=cfg.reliability_threshold, ref_label='reliability threshold', color_by='family'),
    'i1_reliability_forest', {'reliability': RP_}, 'Which metrics are measured reliably?', 'ICC(1,1) of per-sequence values (kind-centred for pooled drills).')
bcols = [RG.combine_column(m, 'agility') for m in RG.BRAKING_CORE]
MKB = MK.filter(pl.col('column').is_in(bcols)).with_columns(pl.col('column').map_elements(label_col, return_dtype=pl.String)) if MK.height else MK
if MKB.height:
    fig(F.lines(MKB, 'k', 'icc1_mean', 'column', 'Braking-signature reliability at equal trial counts', 'trials per athlete (k)', 'ICC(1,1), mean over draws'),
        'i1_matched_k_braking', {'matched_k': MKB}, 'Is reliability comparable across metrics when every athlete has the same number of trials?',
        'Fixed athlete set; k trials drawn without replacement, repeated.')

In [ ]:
banner('I2', 'Noise and smoothing sensitivity', 'Do athlete rankings survive realistic tracking noise and a different smoothing?')

def _i2(ctx):
    ids, rows = MT.robustness_rows(exp.bundle.meta, cfg.robustness_athletes, cfg.seed)
    base = MT.athlete_table(ST, EVS, rows)
    pert, hows = {}, []
    for k, g in enumerate(MT.perturbation_grid(cfg.noise_sd, cfg.smooth_windows)):
        ev, sq, how = MT.cached_perturbed(RUN, exp, P, rows, g, cfg.seed + 101 * (k + 1), reuse=cfg.reuse_artifacts)
        hows.append(how)
        pert[g['name']] = MT.athlete_table(MT.sequence_table(ev, sq, exp.folds_by_athlete), ev)
    long, summ = MR.sensitivity(base, pert, CAND)
    save(long, 'perturbation_sensitivity', 'reliability')
    save(summ, 'perturbation_summary', 'reliability')
    ctx['cache'] = 'hit' if hows and all(h == 'reused cache' for h in hows) else None
    print(len(ids), 'athletes ·', len(rows), 'Combine sequences re-extracted per perturbation ·', hows)
    return long, summ

SENS_L, SENS_S = RUN.run_stage('I2', _i2, note='re-extraction under noise / smoothing (cached)')
if SENS_L is not None and SENS_L.height:
    H = SENS_L.with_columns(pl.col('column').map_elements(label_col, return_dtype=pl.String))
    fig(F.heatmap(H, 'column', 'perturbation', 'spearman', 'Rank stability under perturbation', fmt='{:.2f}', vmin=0.0, vmax=1.0,
                  subtitle=f'Spearman between baseline and perturbed athlete values · robust if ≥ {cfg.robust_spearman}'),
        'i2_perturbation_heatmap', {'sensitivity': SENS_L}, 'Which metrics are fragile to 10 Hz noise or smoothing?', 'Athlete-level Spearman per perturbation.')

In [ ]:
banner('I3', 'Coverage, missingness and the measurement status', 'Which metrics are measured reliably, robustly and often enough to be candidates?')

def _i3(ctx):
    a = exp.bundle.athletes
    tid = set(int(i) for i in a.filter(pl.col('linked').fill_null(False) & pl.col(TRANSFER_TARGET).is_not_null())['nfl_id'].to_list())
    cov = MR.coverage(ATH, CAND, tid)
    st = MR.measurement_status(REL_T, REL_A, SENS_S if SENS_S is not None else pl.DataFrame(), cov, cfg)
    save(cov, 'coverage', 'reliability')
    save(st, 'measurement_status', 'reliability')
    return cov, st

COV, MSTATUS = RUN.run_stage('I3', _i3, raise_errors=True)
show(MSTATUS.select('column', 'measurement_status', 'reliability_estimator', 'reliability', 'reliability_lo', 'min_perturbation_spearman', 'coverage', 'reasons'),
     digits=3, max_rows=60, status='measurement_status')
fig(MF.coverage_bars(COV, 'Coverage of each metric among transfer athletes'), 'i3_coverage', {'coverage': COV}, 'How often is each metric observed?',
    'Share of linked athletes with the frozen target who have a value.')
n = MSTATUS.group_by('measurement_status').len()
say('Measurement status counts: ' + ', '.join(f'{r[0]} {r[1]}' for r in n.iter_rows()) + '. Higher reliability does NOT imply predictive value — that is tested in Parts L–M.', 'finding')

---
# Part J — Novelty beyond official Combine testing

A reliable metric is still useless for scouting if the stopwatch already contains it. Novelty = 1 − out-of-fold R² of the metric from M1 (+ position), with each fold's model fitted on training athletes only; no NFL outcome is involved. Redundancy is measured against the other M14 metrics and against the existing M12 / M13 features (biomech `c_*`, engineered F0 `eng*`).

In [ ]:
banner('J1', 'Novelty and redundancy', 'Which metrics carry information that official tests and earlier features do not?')

def _j1(ctx):
    global DATA, OFFICIAL_MODS, ENC_NAMES
    DATA, OFFICIAL_MODS, ENC_NAMES = TR.analysis_frame(exp, ATH, READY)
    seq_bio, how = handoff.sequence_metrics(exp, RUN, CFG13)
    ath_bio = biomech.athlete_features(seq_bio)
    bio = [c for c in ath_bio.columns if c.startswith('c_') and not c.endswith('__n_seq')]
    nf = DATA.join(ath_bio.select('nfl_id', *bio), on='nfl_id', how='left')
    existing = [*bio, *[c for c in DATA.columns if c.startswith('eng')]]
    nov = NV.novelty_table(nf, CAND, existing, cfg.novelty_max_r2)
    mat = NV.spearman_matrix(DATA, CAND)
    prio = {r['column']: (r['reliability'] if r['reliability'] is not None else float('nan')) for r in MSTATUS.iter_rows(named=True)}
    clu = NV.redundancy_clusters(mat, cfg.redundancy_rho, prio)
    for n_, d_ in (('novelty', nov), ('spearman_matrix', mat), ('redundancy_clusters', clu)):
        save(d_, n_, 'novelty')
    print('biomech features:', how, '·', DATA.height, 'transfer athletes in the analysis frame')
    return nov, mat, clu

NOV, MAT, CLU = RUN.run_stage('J1', _j1, raise_errors=True)
show(NOV.select('column', 'n', 'oof_r2_m1', 'oof_r2_m1_position', 'novelty', 'novel', 'max_rho_existing', 'max_rho_existing_feature').sort('novelty', descending=True), digits=3, max_rows=60)
NR = NOV.join(MSTATUS.select('column', 'reliability', 'measurement_status'), on='column', how='left').with_columns(pl.col('column').map_elements(label_col, return_dtype=pl.String).alias('label'))
fig(F.scatter(NR.drop_nulls(['novelty', 'reliability']), 'reliability', 'novelty', 'Reliable AND novel? The quality map of M14 metrics', 'trial reliability (ICC(1,1))',
              '1 − OOF R² from M1 + position', group='measurement_status', label='label', ref_x=cfg.reliability_threshold, ref_y=1 - cfg.novelty_max_r2, annotate_top=8),
    'j1_reliability_novelty_map', {'quality': NR}, 'Which metrics are both measurable and not already in the stopwatch?', 'Upper-right = reliable and novel.')
MB = MAT.filter(pl.col('a').is_in(bcols) & pl.col('b').is_in(bcols)).with_columns(pl.col('a').map_elements(label_col, return_dtype=pl.String), pl.col('b').map_elements(label_col, return_dtype=pl.String))
if MB.height:
    fig(F.heatmap(MB, 'a', 'b', 'rho', 'Redundancy inside the braking signature', fmt='{:.2f}', vmin=-1.0, vmax=1.0, diverging=True),
        'j1_braking_redundancy', {'matrix': MB}, 'Are the braking metrics distinct from each other?', 'Spearman between athlete braking metrics.')
show(CLU.filter(pl.col('cluster_size') > 1), max_rows=40)

---
# Part K — The Hidden Braking Advantage

**Hypothesis.** Among prospects with near-identical official tests (stopwatch twins), the one who brakes later and harder, keeps more speed through the cut and re-accelerates sooner shows better NFL movement outcomes.

**Design (outcome-blind until step 3).** (1) A braking signature from the pre-declared braking-core metrics that passed the measurement screen, standardised within position. (2) Twins: same position, Euclidean distance ≤ caliper on within-position z of the official tests; greedy one-to-one matching. Matching sees only official tests. (3) Only then are NFL outcomes joined: the frozen target and the NFL-window braking metrics. (4) Hidden variation: braking differences within twins vs random same-position pairs.

In [ ]:
banner('K1', 'Stopwatch twins and their braking signatures', 'Among athletes with the same stopwatch, do braking signatures differ — and does the better braker do better in the NFL?')

def _k1(ctx):
    global SIGCOLS, SIG, TWCOV, PAIRS, TW_LONG, TW_SUM, HIDDEN, OUTC
    SIGCOLS = SC.signature_columns(MSTATUS, 'agility')
    if not SIGCOLS:
        raise StageSkipped('blocked: no braking-core metric passed the measurement screen')
    SIG = SC.braking_signature(ATH, SIGCOLS)
    a = exp.bundle.athletes
    offs = a.select('nfl_id', *[c for c in cfg.twin_covariates if c in a.columns])
    TWCOV = SC.twin_covariates(offs, cfg.twin_covariates, cfg.twin_min_coverage)
    check(len(TWCOV) >= 2, f'too few official covariates for matching: {TWCOV}')
    PAIRS = SC.stopwatch_twins(SIG.join(offs, on='nfl_id', how='left'), TWCOV, cfg.twin_caliper)
    # ---- outcomes are joined only AFTER matching ----
    ncols = [RG.nfl_column(m) for m in RG.BRAKING_CORE if RG.nfl_column(m) in ATH.columns]
    OUTC = a.select('nfl_id', TRANSFER_TARGET).join(ATH.select('nfl_id', *ncols), on='nfl_id', how='left')
    TW_LONG, TW_SUM = SC.twin_contrast(PAIRS, SIG, OUTC, [TRANSFER_TARGET, *ncols], cfg.n_boot, cfg.seed)
    HIDDEN = SC.hidden_variation(PAIRS, SIG, 4000, cfg.seed)
    for n_, d_ in (('braking_signature', SIG), ('stopwatch_twins', PAIRS), ('twin_contrast_pairs', TW_LONG), ('twin_contrast_summary', TW_SUM), ('hidden_variation', HIDDEN)):
        save(d_, n_, 'braking')
    RUN.save_json({'signature_columns': SIGCOLS, 'twin_covariates': TWCOV, 'caliper': cfg.twin_caliper, 'n_pairs': PAIRS.height}, 'hba_design')
    return PAIRS.height

NP = RUN.run_stage('K1', _k1, note='signature · twins · outcome-blind matching · contrast')
if NP:
    print('signature metrics:', list(SIGCOLS), '· twin covariates:', TWCOV, '·', NP, 'twin pairs')
    show(TW_SUM, digits=4)
    show(HIDDEN, digits=3)
    fig(F.forest(TW_SUM.filter(pl.col('mean_diff_hi_minus_lo').is_not_null()), 'outcome', 'mean_diff_hi_minus_lo', 'ci_lo', 'ci_hi',
                 'Better braker minus worse braker within stopwatch twins', 'mean within-pair difference (95 % pair-bootstrap CI)', ref=0.0, ref_label='no difference'),
        'k1_twin_contrast', {'summary': TW_SUM}, 'Does the better braking twin show better NFL movement outcomes?', 'Pairs matched on official tests only; outcomes revealed after matching.')
    fig(F.hbar(HIDDEN.with_columns(pl.col('metric').str.replace('agility__', '')), 'metric', 'ratio_twin_to_random', 'Braking variation the stopwatch leaves unexplained',
               'within-twin |Δz| / random same-position |Δz|', fmt='{:.2f}', ref=1.0, subtitle='≈ 1: twins differ in braking as much as random athletes of the position'),
        'k1_hidden_variation', {'hidden': HIDDEN}, 'How much braking variation remains among stopwatch twins?', 'Ratio of within-twin to random-pair differences.')
    dcol = f'delta_{TRANSFER_TARGET}'
    if TW_LONG.height and dcol in TW_LONG.columns:
        fig(F.scatter(TW_LONG.drop_nulls(dcol), 'delta_composite', dcol, 'Braking difference vs NFL outcome difference within twins', 'Δ braking composite (z)',
                      'Δ nfl.q90_shared (better − worse braker)', ref_y=0.0),
            'k1_dcomposite_vs_doutcome', {'pairs': TW_LONG}, 'Does a larger braking gap go with a larger outcome gap?', 'One point per twin pair.')
    r0 = TW_SUM.filter(pl.col('outcome') == TRANSFER_TARGET)
    if r0.height and r0['ci_lo'][0] is not None:
        lo, hi = r0['ci_lo'][0], r0['ci_hi'][0]
        verdict = 'supports' if lo > 0 else ('contradicts' if hi < 0 else 'does not resolve')
        say(f'Twin contrast on the frozen target: mean {r0["mean_diff_hi_minus_lo"][0]:+.4f}, 95 % CI [{lo:+.4f}, {hi:+.4f}] over {r0["n_pairs"][0]} pairs — this {verdict} the Hidden Braking Advantage. Twins are matched on measured tests only (residual confounding); this is an association, not a causal effect.', 'finding')

---
# Part L — Position-specific NFL transfer

The decisive test: out-of-fold prediction of `nfl.q90_shared` on athlete-disjoint outer folds with the **official engine** (same analysis frame, folds, seed, ridge, inner-CV α, position dummies in every model). M1, M1 + F0 and M1 + combo encoder are refitted inside the M14 run and must reproduce the official ladder exactly (asserted); all comparisons are made on the M13 **strict common cohort**. M14 feature selection is outcome-free and fitted on training athletes only.

In [ ]:
banner('L1', 'The official ladder, baselines and the M14 ladder', 'Do the M14 metric families add held-out predictive value beyond M1?')

def _l1(ctx):
    global OOF, BASE, F0IDS, M14OOF, SELLOG, LADLOG, ELIGIBLE
    OOF = TF.ladder_predictions(exp, READY, reuse=cfg.reuse_artifacts)
    BASE, _ = TF.baseline_predictions(exp)
    rc = TF.reproducibility_check(OOF, BASE)
    check(rc['status'] == 'pass', f'baseline M1 does not reproduce the ladder M1: {rc}')
    F0IDS = TF.f0_available(DATA)
    ELIGIBLE = MSTATUS.filter(pl.col('measurement_status').is_in(['validated', 'exploratory']))['column'].to_list()
    sig = {'transform': M14C.TRANSFORM_VERSION, 'extraction': MT.extraction_signature(exp.bundle.fingerprint, P), 'ready': READY, 'combo': cfg.combo_encoder,
           'thr': cfg.reliability_threshold, 'cov': cfg.min_coverage, 'nov': cfg.novelty_max_r2, 'rho': cfg.redundancy_rho, 'min': cfg.min_reliability_athletes,
           'single': cfg.single_metric_models, 'eligible': sorted(ELIGIBLE), 'cand': sorted(CAND), 'science': exp.cfg.science_hash()}
    prov = {'signature': M14C.params_hash(sig), 'kind': 'm14_ladder'}
    hit = RUN.load_data('datasets/cached/m14_ladder_oof', prov) if cfg.reuse_artifacts else None
    if hit is not None and RUN.has('selection_log', 'transfer'):
        M14OOF, SELLOG = hit, RUN.frame('selection_log', 'transfer')
        LADLOG = RUN.frame('ladder_log', 'transfer') if RUN.has('ladder_log', 'transfer') else pl.DataFrame()
        ctx['cache'] = 'hit'
    else:
        log = []
        mods = TR.build_models(_backend.ae(), DATA, OFFICIAL_MODS, CAND, CACHE, cfg, ELIGIBLE, log)
        print('fitting', len(mods), 'models:', list(mods)[:12], '…')
        M14OOF, SELLOG, LADLOG = TR.run_ladder(exp, DATA, mods, log)
        RUN.data_file(M14OOF, 'datasets/cached/m14_ladder_oof', prov)
        save(SELLOG, 'selection_log', 'transfer')
        save(LADLOG, 'ladder_log', 'transfer')
    RUN.data_file(OOF, 'datasets/cached/official_ladder_oof', {'models': READY, 'kind': 'transfer_oof_v1'})
    save(pl.DataFrame([rc]), 'baseline_reproducibility', 'transfer')
    return M14OOF.height

RUN.run_stage('L1', _l1, note='official ladder + POS/MEAN + M14 ladder (cached)', raise_errors=True)
print(M14OOF.height, 'athletes ·', len([c for c in M14OOF.columns if c.startswith('pred_')]), 'M14-run models')

In [ ]:
banner('L2', 'Alignment, the common cohort and held-out metrics', 'On the same held-out athletes, which models beat M1?')

def _l2(ctx):
    global INT, LONG, COH, MODELS, EV_MAIN, OFFICIAL_MODELS
    OFFICIAL_MODELS = [m for m in ['M1', 'M1+F0', *[f'M1+{n}' for n in READY]] if f'pred_{m}' in OOF.columns]
    INT = TR.integrate(OOF, BASE, M14OOF, OFFICIAL_MODELS, cfg.combo_encoder, exp.folds_by_athlete, F0IDS, READY)
    save(INT['repro'], 'm14_reproducibility', 'transfer')
    save(INT['audit'], 'alignment_audit', 'transfer')
    check(INT['repro'].filter(pl.col('status') != 'pass').height == 0, 'the M14 run does not reproduce the official ladder: every comparison is refused')
    LONG, COH = INT['long'], dict(INT['cohorts'])
    COH['m14_observed'] = TR.observed_cohort(COH['strict_common'], ATH, list(SIGCOLS))
    check(len(COH['strict_common']) >= 50, f'strict common cohort has {len(COH["strict_common"])} athletes (< 50)')
    desc = pl.concat([INT['cohort_desc'], pl.DataFrame([{'cohort': 'm14_observed', 'n_athletes': len(COH['m14_observed']),
                                                         'definition': 'strict_common ∩ every braking-signature metric observed (secondary)'}])], how='diagonal_relaxed')
    save(desc, 'cohorts', 'transfer')
    RUN.data_file(LONG, 'datasets/cached/m14_long_predictions', {'kind': 'long_predictions'})
    MODELS = ['MEAN', 'POS', *OFFICIAL_MODELS, *INT['m14_models']]
    EV_MAIN = TR.evaluate(LONG, COH['strict_common'], MODELS, 'strict_common', cfg.combo_encoder, cfg.n_boot, cfg.seed)
    for k in ('metrics', 'paired', 'intervals', 'folds'):
        save(EV_MAIN[k], f'strict_{k}', 'transfer')
    save(EV_MAIN['wide'], 'strict_fold_predictions', 'transfer')
    return desc

COH_DESC = RUN.run_stage('L2', _l2, raise_errors=True)
show(INT['repro'], status='status')
show(COH_DESC)
KEY = ['POS', 'M1', 'M1+F0', *[m for m in MODELS if m.startswith('M1+M14') or m == f'M1+{TR.combo_name(cfg.combo_encoder)}']]
show(EV_MAIN['metrics'].filter(pl.col('model').is_in(KEY)).select('model', 'n_athletes', 'rmse', 'mae', 'r2', 'r2_within_position', 'skill_vs_position_oof', 'fold_rmse_sd'), digits=4)
PV = EV_MAIN['paired'].filter((pl.col('reference') == 'M1') & pl.col('model').is_in(KEY)).with_columns(pl.col('model').map_elements(F.short, return_dtype=pl.String).alias('label')).sort('delta_rmse')
fig(F.forest(PV, 'label', 'delta_rmse', 'delta_lo', 'delta_hi', 'Held-out gain over M1 on the strict common cohort', 'Δ RMSE vs M1 (positive = better), fold-stratified bootstrap 95 % CI',
             ref=0.0, ref_label='M1'),
    'l2_delta_vs_m1', {'paired': PV}, 'Which M14 families predict NFL movement beyond M1?', 'Paired athlete bootstrap on one common cohort.')

In [ ]:
banner('L3', 'Position-specific transfer', 'Does any gain hold inside positions, rather than through position composition?')

def _l3(ctx):
    pm = [m for m in KEY if m not in ('MEAN',)]
    m, p = TR.by_position(EV_MAIN['wide'], pm, cfg.min_position_n, NBOOT_FAST, cfg.seed)
    save(m, 'position_metrics', 'position')
    save(p, 'position_paired', 'position')
    return m, p

POS_M, POS_P = RUN.run_stage('L3', _l3)
if POS_P is not None and POS_P.height:
    PP = POS_P.with_columns(pl.col('model').map_elements(F.short, return_dtype=pl.String))
    fig(F.heatmap(PP, 'model', 'position_group', 'delta_rmse', 'Δ RMSE vs M1 inside each position', fmt='{:+.3f}', diverging=True, center=0.0,
                  subtitle=f'positions with ≥ {cfg.min_position_n} athletes in the strict cohort · exploratory (BH inside each position)'),
        'l3_position_heatmap', {'paired': POS_P}, 'Where (which positions) does movement information help?', 'Within-position paired Δ RMSE vs M1.')
    show(POS_P.select('position_group', 'model', 'n_athletes', 'delta_rmse', 'delta_lo', 'delta_hi', 'verdict', 'p_bh_all'), digits=4, status='verdict', max_rows=60)
    say('Position-specific results use small samples and are exploratory; they generate hypotheses for position-specific scouting, they do not validate them.', 'caveat')

---
# Part M — Statistical comparisons and robustness

Declared predictive status (per model vs M1): **validated** = CI lower bound > 0, adjusted p ≤ 0.05 (Holm in the primary family, BH otherwise) and better in ≥ 4 of 5 folds; **exploratory** = positive Δ with bootstrap share Δ ≤ 0 below 0.2; **invalid** = CI upper bound < 0 (reliably worse); otherwise **inconclusive**. Robustness: fold stability, secondary cohorts, unstratified bootstrap, selection stability across folds and a within-position permutation null.

In [ ]:
banner('M1', 'Statuses, fold stability and secondary cohorts', 'Are the conclusions stable across folds, cohorts and resampling schemes?')

def _m1(ctx):
    global STAB, STATUS_ALL, PRIMARY_STATUS, FAMILY_STATUS, SINGLE_STATUS, ROBUST, SELFREQ
    prim = TR.primary_models(cfg.combo_encoder)
    STAB = TR.fold_stability(EV_MAIN['folds'])
    STATUS_ALL = TR.transfer_status(EV_MAIN['paired'], STAB)
    vs1 = STATUS_ALL.filter(pl.col('reference') == 'M1')
    PRIMARY_STATUS = vs1.filter(pl.col('model').is_in(prim))
    FAMILY_STATUS = vs1.filter(pl.col('model').str.starts_with('M1+M14_') & ~pl.col('model').is_in(prim))
    SINGLE_STATUS = vs1.filter(pl.col('model').str.starts_with('M1+m14c_'))
    rows = [PRIMARY_STATUS.with_columns(pl.lit('primary · strict_common · fold-stratified').alias('check'))]
    for coh in ('m14_observed', 'ladder_all'):
        ids = COH.get(coh, set())
        if len(ids) >= 30:
            e = TR.evaluate(LONG, ids, MODELS, coh, cfg.combo_encoder, NBOOT_FAST, cfg.seed)
            save(e['paired'], f'{coh}_paired', 'robustness')
            s = TR.transfer_status(e['paired'], TR.fold_stability(e['folds']))
            rows.append(s.filter((pl.col('reference') == 'M1') & pl.col('model').is_in(prim)).with_columns(pl.lit(f'secondary cohort {coh} (n={len(ids)})').alias('check')))
    pp = [(m, 'M1') for m in prim if m in MODELS]
    un = TF.paired_comparisons(EV_MAIN['wide'], pp, cfg.n_boot, cfg.seed, 'strict_common', stratify=False, primary_family=pp)
    rows.append(TR.transfer_status(un, STAB).with_columns(pl.lit('unstratified athlete bootstrap').alias('check')))
    ROBUST = pl.concat(rows, how='diagonal_relaxed')
    SELFREQ = TR.selection_frequency(SELLOG)
    for n_, d_ in (('transfer_status_all', STATUS_ALL), ('primary_status', PRIMARY_STATUS), ('family_status', FAMILY_STATUS), ('single_metric_status', SINGLE_STATUS),
                   ('fold_stability', STAB), ('robustness_summary', ROBUST), ('selection_frequency', SELFREQ)):
        save(d_, n_, 'robustness')
    return ROBUST

RUN.run_stage('M1', _m1, raise_errors=True)
show(PRIMARY_STATUS, digits=4, status='status')
show(FAMILY_STATUS, digits=4, status='status')
show(ROBUST.select('check', 'model', 'delta_rmse', 'delta_lo', 'delta_hi', 'p_adjusted', 'status'), digits=4, status='status')
show(SINGLE_STATUS.sort('delta_rmse', descending=True), digits=4, status='status', max_rows=40)
FD = EV_MAIN['folds'].filter(pl.col('model').is_in(['M1', *TR.primary_models(cfg.combo_encoder)])).with_columns(pl.col('model').map_elements(F.short, return_dtype=pl.String))
fig(F.fold_points(FD, 'model', 'fold', 'delta_rmse_vs_M1', 'Fold-level Δ RMSE vs M1 of the primary models', 'Δ RMSE vs M1 (positive = better)', ref=0.0),
    'm1_fold_stability', {'folds': FD}, 'Is any gain carried by a single fold?', 'One point per outer fold.')
if SELFREQ.height:
    show(SELFREQ.filter(pl.col('model').is_in(TR.primary_models(cfg.combo_encoder))), digits=2, max_rows=40)

In [ ]:
banner('M2', 'Within-position permutation null', 'Would randomly re-assigned M14 values (same dimension, same selection) give the same gain?')

def _m2(ctx):
    if cfg.n_permutations <= 0:
        raise StageSkipped('n_permutations = 0')
    obs = EV_MAIN['paired'].filter((pl.col('model') == 'M1+M14_reliable') & (pl.col('reference') == 'M1'))
    if not obs.height:
        raise StageSkipped('blocked: M1+M14_reliable not in the comparison')
    sig = {'ladder': M14C.params_hash(sorted(M14OOF.columns)), 'n': cfg.n_permutations, 'seed': cfg.seed, 'cand': sorted(CAND), 'cohort': len(COH['strict_common'])}
    hit = RUN.cached('robustness', ['permutation_null'], sig, 'robustness/permutation')
    if hit is not None:
        null = hit['permutation_null']
        ctx['cache'] = 'hit'
    else:
        null = TR.permutation_null(exp, DATA, CAND, CACHE, cfg, EV_MAIN['wide'], cfg.n_permutations, cfg.seed + 7)
        RUN.save_all({'permutation_null': null}, 'robustness', sig, 'robustness/permutation')
    p = TR.permutation_p(null, float(obs['delta_rmse'][0]))
    RUN.save_json(p, 'permutation_test')
    return null, p

PERM = RUN.run_stage('M2', _m2, note='within-position permutation null (cached)')
if PERM:
    NULL, PP_ = PERM
    print(PP_)
    fig(F.histogram_by(NULL.with_columns(pl.lit('permuted').alias('g')), 'delta_rmse', 'g', 'Permutation null of M1 + M14_reliable',
                       f'Δ RMSE vs M1 under within-position permutation · observed {PP_["observed_delta"]:+.4f}', bins=20),
        'm2_permutation_null', {'null': NULL}, 'Is the observed gain larger than chance features would give?', f'p_perm = {PP_["p_perm"]:.3f} with {PP_["n_permutations"]} permutations.')

---
# Part N — Scouting case studies

Pairs are selected **without outcomes**: the largest braking-composite gap among the closest stopwatch twins. Each case study (`reports/athlete_profiles/case_study_XX.md|.html`) shows the official tests, the braking signature with within-position percentiles, the event-aligned speed / acceleration profiles, and — revealed after selection — the NFL outcomes, with the interpretation limits. Athletes are pseudonymised.

In [ ]:
banner('N1', 'Anonymised stopwatch-twin case studies', 'What does a hidden braking difference look like for two athletes with the same stopwatch?')

def _n1(ctx):
    if not TW_LONG.height:
        raise StageSkipped('blocked: no twin pairs (Part K)')
    cases = SC.select_case_studies(TW_LONG, cfg.n_case_studies)
    ids = cases['athlete_hi'].to_list() + cases['athlete_lo'].to_list()
    lab = labels_for(ids)
    a = exp.bundle.athletes
    card = ATH.join(a.select('nfl_id', *[c for c in {*TWCOV, TRANSFER_TARGET} if c in a.columns]), on='nfl_id', how='left')
    ocols = [TRANSFER_TARGET, *[RG.nfl_column(m) for m in RG.BRAKING_CORE if RG.nfl_column(m) in ATH.columns]]
    idx = []
    for k, r in enumerate(cases.iter_rows(named=True), 1):
        prof = MF.twin_profile(EVS, [r['athlete_hi'], r['athlete_lo']], lab, P.trace_half_window)
        out = fig(MF.event_anatomy(prof, f'Case {k}: {lab[r["athlete_hi"]]} vs {lab[r["athlete_lo"]]} ({r["position_group"]})', 'agility cuts, complete events, aligned on the minimum'),
                  f'n1_case_{k:02d}', {'profile': prof}, 'How do the two twins brake and re-accelerate?', 'Median ± IQR of each athlete’s agility cuts.')
        md = SC.case_study_markdown(k, r, card, SIG, TWCOV, list(SIGCOLS), ocols, lab, figure_rel=('../../' + out['png']) if out.get('png') else None)
        RP.write_profile(RUN, md, f'case_study_{k:02d}')
        idx.append({'case': k, 'position_group': r['position_group'], 'athlete_hi': lab[r['athlete_hi']], 'athlete_lo': lab[r['athlete_lo']],
                    'official_distance': r['distance'], 'delta_composite': r['delta_composite'], 'profile': f'reports/athlete_profiles/case_study_{k:02d}.html'})
    t = pl.DataFrame(idx)
    save(t, 'case_study_index', 'scouting')
    return t

CASES_T = RUN.run_stage('N1', _n1)
show(CASES_T, digits=3)
say('Case studies illustrate the measured contrast; they are not evidence on their own and are never scouting guidance.', 'caveat')

---
# Part O — Final scientific synthesis and export

The evidence table joins, per metric: measurement status, novelty, single-metric and family transfer status, a final status by the declared rule, the scouting hypothesis it supports (worded by its status) and the validation still required. The final report (`reports/markdown|html/m14_final_report`) is generated from the saved tables only.

In [ ]:
banner('O1', 'Evidence synthesis and the answers to the research questions', 'Which features are reliable, novel and predictive — and what scouting hypothesis follows?')

def _o1(ctx):
    global EVID, ANS
    EVID = SC.evidence_table(AV, MSTATUS, NOV, SINGLE_STATUS if 'SINGLE_STATUS' in globals() else None, FAMILY_STATUS if 'FAMILY_STATUS' in globals() else None)
    ANS = pl.DataFrame(SC.research_answers(EVID, PRIMARY_STATUS if 'PRIMARY_STATUS' in globals() else None, TW_SUM))
    save(EVID, 'evidence_table', 'synthesis')
    save(ANS, 'research_answers', 'synthesis')
    return EVID

RUN.run_stage('O1', _o1, raise_errors=True)
for r in ANS.iter_rows(named=True):
    display(Markdown(f'**{r["question"]}** {r["answer"]}'))
show(EVID.select('metric', 'scope', 'family', 'measurement_status', 'novel', 'single_metric_status', 'family_status', 'final_status'), status='final_status', max_rows=80)
EV_FIG = EVID.sort('final_status', 'family')
fig(MF.status_matrix(EV_FIG, 'Evidence ladder of every M14 metric'), 'o1_status_matrix', {'evidence': EV_FIG}, 'Where does each metric stop on the evidence ladder?',
    'measurement → single-metric transfer → family transfer → final status.')

In [ ]:
banner('O2', 'Final report, provenance and export', 'Is every result saved, reproducible and reloadable?')

def _o2(ctx):
    from nflbowl.utils.reproducibility import utc_now
    prov = pl.DataFrame([{'key': k, 'value': str(v)} for k, v in {**RUN.provenance, 'ready_encoders': READY, 'combo_encoder': TR.combo_name(cfg.combo_encoder),
                                                                  'strict_common': len(COH.get('strict_common', [])), 'event_params': P.to_dict()}.items()])
    save(prov, 'provenance', 'synthesis')
    metrics_key = EV_MAIN['metrics'].filter(pl.col('model').is_in(KEY))
    rep = {'run_dir': str(RUN.root), 'bundle': exp.bundle.fingerprint, 'science': exp.cfg.science_hash(), 'version': nflbowl.__version__, 'generated_at': utc_now(),
           'primary_status': PRIMARY_STATUS, 'cohort_desc': COH_DESC, 'metrics': metrics_key, 'availability_summary': AV_SUM, 'feasibility': QUAL,
           'mstatus': MSTATUS, 'novelty': NOV, 'twin_summary': TW_SUM, 'hidden_variation': HIDDEN if 'HIDDEN' in globals() else None,
           'family_status': FAMILY_STATUS, 'single_status': SINGLE_STATUS, 'position_summary': POS_P if POS_P is not None else None,
           'robustness': ROBUST.select('check', 'model', 'delta_rmse', 'delta_lo', 'delta_hi', 'status'), 'answers': ANS, 'evidence': EVID, 'provenance': prov}
    md = RP.final_report(rep)
    RP.write_report(RUN, md, 'm14_final_report')
    final = {'primary': PRIMARY_STATUS.to_dicts(), 'answers': ANS.to_dicts(), 'n_metrics': EVID.height,
             'status_counts': dict(EVID.group_by('final_status').len().iter_rows()), 'generated_at': utc_now()}
    RUN.save_json(final, 'final_results')
    return md

MD = RUN.run_stage('O2', _o2)
RUN.write_status({'final_report': 'reports/html/m14_final_report.html'})
st = RUN.stage_records()
show(pl.DataFrame([{'stage': k, 'status': v.get('status'), 'seconds': v.get('seconds'), 'cache': v.get('cache'), 'note': v.get('note')} for k, v in st.items()]), status='status', max_rows=60)
try:
    z = RUN.export_zip()
    print('export:', z)
except Exception as e:
    print('zip export failed (results are still on Drive):', e)
if MD:
    display(Markdown(MD[:6000]))